# 03 · Feature Engineering ба Selection

**Feature** гэдэг нь model-д өгөх "сэжүүр". Бид өнөөдрийг хүртэлх мэдээллээс сэжүүр гаргаж,
**ирэх 3 өдрийн** анхаарлыг таамаглана.

| Алхам | Юу хийх |
|---|---|
| Extraction | Огнооноос гараг, сар, улирлыг ялгаж авах |
| Engineering | Өнгөрсөн өдрүүдээс шинэ хувьсагч үүсгэх: lag, rolling, өсөлт, оргил, түвшин |
| Selection | Давхардсан, хэрэггүй feature-ийг хасах |

> ⚠️ **Хамгийн чухал дүрэм:** feature-т **ирээдүйн** мэдээлэл орох ёсгүй (data leakage).
> Үүнийг `tests/test_features.py` доторх 5 тестээр шалгасан.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px, plotly.io as pio
import lightgbm as lgb
warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"
BLUE, RED, GRAY = "#2a78d6", "#e63946", "#9aa0ad"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import clean
from src.features import make_features, FEATURES_BASE, GROUPS, split_of

P = ROOT / "data" / "processed"
cl = pd.read_parquet(P / "clean.parquet") if (P / "clean.parquet").exists() else clean.build()

## 1. Feature үүсгэх

"Таамаг гаргах өдөр" **t** гэж үзье. Бид t өдрийг оролцуулан өмнөх бүхнийг мэднэ:

| Бүлэг | Feature | Энгийнээр |
|---|---|---|
| lag | `lag_1, 2, 3, 7, 14` | 1, 2, 3, 7, 14 өдрийн өмнөх утга |
| rolling | `roll_mean_*, roll_std_*, roll_max_28` | Сүүлийн 3/7/14/28 хоногийн дундаж, хэлбэлзэл, оргил |
| өсөлт | `growth_1, growth_3, ratio_7, from_max_28` | Өсч байна уу, буурч байна уу; 7 хоногийн дунджаас хэр их вэ |
| оргил | `days_since_peak` | Сүүлийн оргилоос хойш хэдэн хоног өнгөрсөн |
| түвшин | `level_365, vs_level` | Сэдвийн "ердийн" түвшин ба өнөөдөр түүнээс хэр дээр вэ |
| ангилал | `category` | Улс төр, кино, спорт… (сэдвийн **нэр биш**) |
| календарь | `dow, is_weekend, month, doy_sin/cos` | Гараг, амралтын өдөр, сар, жилийн аль үе |

> **Яагаад сэдвийн нэрийг feature болгохгүй вэ?** Нэрийг өгвөл model "Taylor Swift ийм байдаг" гэж цээжилнэ.
> Нэргүй бол хэв маягаас сурч, **урьд өмнө хараагүй** шинэ вирал сэдэв дээр ч ажиллана.

In [2]:
f = make_features(cl[["date", "article", "category", "y"]])
f["split"] = split_of(f["date"])
num = [c for c in FEATURES_BASE if c != "category"]
f = f.dropna(subset=num + ["y_h1", "y_h2", "y_h3"])     # эхний 28+ өдөр (түүх хүрэлцэхгүй) хасагдана
print(f"Feature: {len(FEATURES_BASE)}   Мөр: {len(f):,}")
f[["date", "article", "y_0", "lag_1", "roll_mean_7", "ratio_7", "days_since_peak", "y_h1", "y_h2", "y_h3"]].tail(4)

Feature: 28   Мөр: 574,942


,date,article,y_0,lag_1,roll_mean_7,ratio_7,days_since_peak,y_h1,y_h2,y_h3
583965,2026-10-31,Ögedei_Khan,1.560889,1.710001,1.645365,-0.084476,17.0,1.636853,1.620499,1.622508
583966,2026-11-01,Ögedei_Khan,1.636853,1.560889,1.640296,-0.003443,18.0,1.620499,1.622508,1.530174
583967,2026-11-02,Ögedei_Khan,1.620499,1.636853,1.625813,-0.005314,19.0,1.622508,1.530174,1.587753
583968,2026-11-03,Ögedei_Khan,1.622508,1.620499,1.619958,0.002550,20.0,1.530174,1.587753,1.613892


## 2. Цаг хугацааны хуваалт

Time series өгөгдлийг **санамсаргүй холих хориотой**: холивол model ирээдүйг "шагайж" хуурамчаар сайн харагдана.
Тиймээс он дарааллаар нь хуваана:

In [3]:
sp = f.groupby("split").agg(эхлэх=("date", "min"), дуусах=("date", "max"), мөр=("y", "size")).loc[["train", "valid", "test"]]
sp["хувь"] = (sp["мөр"] / sp["мөр"].sum() * 100).round(1)
sp

,эхлэх,дуусах,мөр,хувь
split,,,,
train,2021-01-28,2025-06-30,427959,74.4
valid,2025-07-01,2026-03-31,81666,14.2
test,2026-04-01,2026-11-03,65317,11.4


In [4]:
cnt = f.groupby([pd.Grouper(key="date", freq="MS"), "split"]).size().rename("мөр").reset_index()
fig = px.bar(cnt, x="date", y="мөр", color="split", color_discrete_map={"train": BLUE, "valid": "#f0a500", "test": RED},
             title="Train → Validation → Test: цаг хугацааны дарааллаар")
fig.update_layout(xaxis_title=None, legend_title=None, bargap=0)
fig.show()

## 3. Юуг таамаглах вэ: түвшин биш, **өөрчлөлт**

Шууд `y_h1`-ийг биш, **өнөөдрөөс хэр өөрчлөгдөхийг** (`y_h1 − y_0`) таамаглана.

> **Яагаад?** Модны алгоритм (LightGBM) сургалтанд хараагүй хэт өндөр/нам утгыг гаргаж чаддаггүй.
> Өөрчлөлт нь бүх сэдэвт ижил хэмжээстэй (жижиг ч, том ч сэдэв "+0.3" өсч болно) тул model илүү ерөнхий дүрэм сурна.
> Таамаг гаргахдаа: **y_h1 = y_0 + таамагласан өөрчлөлт**.

Feature-ийг хурдан үнэлэхийн тулд энгийн LightGBM ашиглана (hyperparameter-ийг маргааш Optuna-аар оновчлоно).
Хурдасгах үүднээс train-ийн сүүлийн 2 жилийг авна.

In [5]:
tr = f[(f["split"] == "train") & (f["date"] >= "2023-07-01")]
va = f[f["split"] == "valid"]

def quick(feats, h=1):
    """Энгийн LightGBM сургаж, validation дээрх MAE-г буцаана (бага = сайн)."""
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, objective="l1", verbose=-1)
    m.fit(tr[feats], tr[f"y_h{h}"] - tr["y_0"], eval_set=[(va[feats], va[f"y_h{h}"] - va["y_0"])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return float(np.mean(np.abs(m.predict(va[feats]) - (va[f"y_h{h}"] - va["y_0"])))), m

naive = float(np.mean(np.abs(va["y_h1"] - va["y_0"])))        # "маргааш өнөөдрийнх шиг"
mae_all, m_all = quick(FEATURES_BASE)
print(f"Энгийн арга (өчигдрийнх шиг): {naive:.4f}")
print(f"LightGBM, бүх {len(FEATURES_BASE)} feature: {mae_all:.4f}  ({(1 - mae_all / naive) * 100:.0f}% бага алдаа)")

Энгийн арга (өчигдрийнх шиг): 0.1119
LightGBM, бүх 28 feature: 0.0973  (13% бага алдаа)


## 4. Feature selection

### 4а. Importance: model аль feature-ийг хамгийн их ашигласан бэ?

In [6]:
imp = pd.Series(m_all.booster_.feature_importance("gain"), index=FEATURES_BASE)
imp = (imp / imp.sum() * 100).sort_values()
fig = px.bar(x=imp.values, y=imp.index, orientation="h", color_discrete_sequence=[BLUE],
             title="Feature importance (gain, %)")
fig.update_layout(xaxis_title="%", yaxis_title=None, height=650)
fig.show()

### 4б. Давхардлыг хасах (корреляц > 0.95)

Хоёр feature бараг ижил мэдээлэл агуулбал (корреляц > 0.95) нэгийг нь хасна.
Аль нь үлдэх вэ? → **importance өндөртэй** нь.

In [7]:
corr = tr[num].corr().abs()
keep, dropped = [], []
for c in imp.sort_values(ascending=False).index:
    twin = [k for k in keep if k != "category" and c != "category" and corr.loc[c, k] > 0.95]
    if twin:
        dropped.append({"хассан": c, "ижил төстэй": twin[0], "корреляц": round(corr.loc[c, twin[0]], 3)})
    else:
        keep.append(c)
pd.DataFrame(dropped)

,хассан,ижил төстэй,корреляц
0,y_0,roll_mean_7,0.984
1,roll_mean_3,roll_mean_7,0.994
2,lag_1,roll_mean_7,0.988
3,lag_7,roll_mean_7,0.978
4,lag_2,roll_mean_7,0.991
5,lag_14,roll_mean_7,0.952
6,lag_3,roll_mean_7,0.991
7,roll_mean_14,roll_mean_7,0.994
8,roll_mean_28,roll_mean_7,0.982


In [8]:
top = ["y_0", "lag_1", "lag_7", "roll_mean_3", "roll_mean_7", "roll_mean_28", "ratio_7", "growth_1", "vs_level"]
fig = px.imshow(corr.loc[top, top].round(2), text_auto=True, color_continuous_scale=["#ffffff", BLUE], zmin=0, zmax=1,
                title="Корреляц: lag ба rolling дундаж бараг ижил (0.95+), харин өсөлтийн feature-үүд өөр мэдээлэлтэй")
fig.show()

In [9]:
mae_keep, m_keep = quick(keep)
print(f"Бүх feature ({len(FEATURES_BASE)}): MAE {mae_all:.4f}")
print(f"Хассаны дараа ({len(keep)}):  MAE {mae_keep:.4f}   -> {len(FEATURES_BASE) - len(keep)} feature хассан ч үр дүн бараг ижил")

Бүх feature (28): MAE 0.0973
Хассаны дараа (19):  MAE 0.0974   -> 9 feature хассан ч үр дүн бараг ижил


### 4в. Ablation: бүлэг бүрийг хасаад үзэх

Нэг бүлгийг бүхэлд нь хасаад MAE хэр **муудаж** байгааг харна. Их муудвал тэр бүлэг чухал.

In [10]:
rows = []
for g, cols in GROUPS.items():
    fs = [c for c in keep if c not in cols]
    if len(fs) == len(keep):
        continue                                   # энэ бүлгээс үлдсэн feature алга
    mae, _ = quick(fs)
    rows.append({"бүлэг": g, "хассан feature": len(keep) - len(fs), "MAE": round(mae, 4), "муудсан": round(mae - mae_keep, 4)})
abl = pd.DataFrame(rows).sort_values("муудсан", ascending=False)
fig = px.bar(abl, x="муудсан", y="бүлэг", orientation="h", color_discrete_sequence=[RED],
             title="Бүлгийг хасахад validation MAE хэр муудсан бэ (их = чухал)")
fig.update_layout(yaxis={"categoryorder": "total ascending"}, yaxis_title=None, xaxis_title="MAE-ийн өсөлт")
fig.show()
abl

,бүлэг,хассан feature,MAE,муудсан
1,өсөлт,4,0.0994,0.0021
5,календарь,5,0.0991,0.0017
4,ангилал,1,0.0986,0.0012
0,rolling,6,0.0979,0.0005
2,оргил,1,0.0975,0.0002
3,түвшин,2,0.0975,0.0001


## 5. Хадгалах

In [11]:
final = keep
(P / "feature_list.json").write_text(json.dumps({
    "features": final, "categorical": ["category"], "target": "y_h{h} - y_0", "horizons": [1, 2, 3],
    "dropped_corr": [d["хассан"] for d in dropped],
    "quick_valid_mae_h1": {"naive": round(naive, 4), "all": round(mae_all, 4), "selected": round(mae_keep, 4)},
    "ablation": abl.to_dict("records"),
}, ensure_ascii=False, indent=2), encoding="utf-8")
cols = ["date", "article", "split", "y_0", "y_h1", "y_h2", "y_h3"] + final
f[cols].to_parquet(P / "features.parquet", index=False, compression="zstd")
print(f"features.parquet: {len(f):,} мөр, {len(final)} feature")
print(final)

features.parquet: 574,942 мөр, 19 feature
['ratio_7', 'vs_level', 'category', 'dow', 'growth_1', 'roll_mean_7', 'roll_std_3', 'roll_std_7', 'growth_3', 'roll_std_28', 'roll_std_14', 'days_since_peak', 'doy_sin', 'doy_cos', 'level_365', 'from_max_28', 'roll_max_28', 'month', 'is_weekend']


## ✅ Дүгнэлт

- **28 feature** үүсгэж, корреляцаар давхардсан **9**-ийг хассан. Үр дүн муудаагүй.
- Хамгийн чухал нь **өсөлтийн** feature-үүд (ratio_7, growth): "өнөөдөр ердийнхөөсөө хэр их вэ".
- **Календарь** ба **ангилал** ч чухал: гараг, улирал, сэдвийн төрөл.
- Хурдан LightGBM ч энгийн "өчигдрийнх шиг" аргаас бага алдаатай.

**Маргааш (04):** Optuna-аар hyperparameter оновчлох, early stopping, overfit шалгалт, test дээр эцсийн үнэлгээ.